<a id="setup"></a>
# Comparaison v1 vs v2 — tokenisation SentencePiece

La v2 a déjà été entraînée dans le module 07 (07a pour le broadcasting, 07b pour le schéma BIO), guidée par les vrais morphèmes de CorBaMa sans jamais s'entraîner sur ses phrases. Ce notebook se contente de recharger les checkpoints existants et de les comparer, sur notre propre test set ET sur CorBaMa — pas de réentraînement ici.

In [4]:
from bambara_pos_utils import *

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from bambara_pos_utils import *

import torch
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

SP_DIR = Path.cwd().parent / "07_sentencepiece_tokenization"

Device : cpu


In [2]:
# Données : même split que les modules 05/06/07 (seed=42), + CorBaMa
corpus_path = Path.cwd().parent / "04_bambara_pos_pipeline" / "bambara_pos_prep_retagged.conll"
corpus_data = load_bambara_corpus(corpus_path)
train_data, val_data, test_data = split_corpus(corpus_data, seed=42)

corbama_data = load_bambara_corpus(Path.cwd().parent / "08_real_data_evaluation" / "corbama_corpus.tsv")
print(f"Test set Bayelemabaga : {len(test_data)} phrases | CorBaMa : {len(corbama_data)} phrases")

Corpus chargé : 37392 phrases.
Corpus chargé : 166361 phrases.
Test set Bayelemabaga : 3740 phrases | CorBaMa : 166361 phrases


<a id="load-broadcast"></a>
# Chargement des modèles — schéma Broadcasting (07a)

In [7]:
sp_v1 = spm.SentencePieceProcessor(model_file=str(SP_DIR / "bambara_sp.model"))
sp_v2 = spm.SentencePieceProcessor(model_file=str(SP_DIR / "bambara_sp_v2.model"))
print("Vocab v1 :", sp_v1.get_piece_size(), "| Vocab v2 :", sp_v2.get_piece_size())

def load_sp_checkpoint(checkpoint_path, sp_processor):
    checkpoint_path = Path(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location=device)

    checkpoint.setdefault(
        "word_to_ix",
        {
            sp_processor.id_to_piece(i): i
            for i in range(sp_processor.get_piece_size())
        },
    )

    patched_path = checkpoint_path.with_suffix(".patched.pth")
    torch.save(checkpoint, patched_path)

    return load_checkpoint(patched_path, device)


model_bc_v1, _, _ = load_sp_checkpoint(
    SP_DIR / "07a_broadcast_scheme" / "models" / "bambara_pos_sp_broadcast.pth",
    sp_v1,
)

model_bc_v2, _, _ = load_sp_checkpoint(
    SP_DIR / "07a_broadcast_scheme" / "models" / "bambara_pos_sp_broadcast_v2.pth",
    sp_v2,
)

print("Modèles broadcasting v1 et v2 chargés.")

Vocab v1 : 4000 | Vocab v2 : 6000
Modèles broadcasting v1 et v2 chargés.


<a id="load-bio"></a>
# Chargement des modèles — schéma BIO (07b)

In [13]:
import bambara_pos_utils as bpu

label2id_bio, id2label_bio = build_bio_tagset(VALID_TAGS)


def load_sp_checkpoint_bio(checkpoint_path, sp_processor):
    checkpoint_path = Path(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location=device)

    checkpoint.setdefault(
        "word_to_ix",
        {
            sp_processor.id_to_piece(i): i
            for i in range(sp_processor.get_piece_size())
        },
    )

    patched_path = checkpoint_path.with_suffix(".patched.pth")
    torch.save(checkpoint, patched_path)

    loader_globals = bpu.load_checkpoint.__globals__
    overrides = {
        "VALID_TAGS": list(label2id_bio.keys()),
        "label2id": label2id_bio,
        "id2label": id2label_bio,
    }
    previous = {
        name: loader_globals.get(name)
        for name in overrides
        if name in loader_globals
    }

    try:
        for name, value in overrides.items():
            if name in loader_globals:
                loader_globals[name] = value
        return bpu.load_checkpoint(patched_path, device)
    finally:
        for name, value in previous.items():
            loader_globals[name] = value


model_bio_v1, _, _ = load_sp_checkpoint_bio(
    SP_DIR / "07b_bio_scheme" / "models" / "bambara_pos_sp_bio.pth",
    sp_v1,
)
model_bio_v2, _, _ = load_sp_checkpoint_bio(
    SP_DIR / "07b_bio_scheme" / "models" / "bambara_pos_sp_bio_v2.pth",
    sp_v2,
)
print("Modèles BIO v1 et v2 chargés.")

Modèles BIO v1 et v2 chargés.


<a id="comparison"></a>
# Comparaison complète : v1 vs v2, sur les deux schémas

In [14]:
def avg_pieces_per_word(sentences, sp):
    total_pieces, total_words = 0, 0
    for words, _ in sentences:
        for w in words:
            total_pieces += len(sp.encode(w, out_type=str))
            total_words += 1
    return total_pieces / total_words

results = {}

# Broadcasting
results["Broadcast v1"] = {
    "acc_own": evaluate_subword_model_word_level(model_bc_v1, test_data, sp_v1, id2label, "broadcast", device)[0],
    "acc_corbama": evaluate_subword_model_word_level(model_bc_v1, corbama_data, sp_v1, id2label, "broadcast", device)[0],
    "fragmentation": avg_pieces_per_word(corbama_data, sp_v1),
}
results["Broadcast v2"] = {
    "acc_own": evaluate_subword_model_word_level(model_bc_v2, test_data, sp_v2, id2label, "broadcast", device)[0],
    "acc_corbama": evaluate_subword_model_word_level(model_bc_v2, corbama_data, sp_v2, id2label, "broadcast", device)[0],
    "fragmentation": avg_pieces_per_word(corbama_data, sp_v2),
}

# BIO
results["BIO v1"] = {
    "acc_own": evaluate_subword_model_word_level(model_bio_v1, test_data, sp_v1, id2label_bio, "bio", device)[0],
    "acc_corbama": evaluate_subword_model_word_level(model_bio_v1, corbama_data, sp_v1, id2label_bio, "bio", device)[0],
    "fragmentation": avg_pieces_per_word(corbama_data, sp_v1),
}
results["BIO v2"] = {
    "acc_own": evaluate_subword_model_word_level(model_bio_v2, test_data, sp_v2, id2label_bio, "bio", device)[0],
    "acc_corbama": evaluate_subword_model_word_level(model_bio_v2, corbama_data, sp_v2, id2label_bio, "bio", device)[0],
    "fragmentation": avg_pieces_per_word(corbama_data, sp_v2),
}

print(f"{'Configuration':<18}{'Acc. propre test':<20}{'Acc. CorBaMa':<18}{'Frag. moyenne'}")
for name, r in results.items():
    print(f"{name:<18}{r['acc_own']*100:.2f}%{'':<12}{r['acc_corbama']*100:.2f}%{'':<10}{r['fragmentation']:.2f}")

Configuration     Acc. propre test    Acc. CorBaMa      Frag. moyenne
Broadcast v1      99.94%            65.60%          1.42
Broadcast v2      99.93%            66.97%          1.39
BIO v1            99.97%            67.14%          1.42
BIO v2            99.98%            65.44%          1.39


In [15]:
# Rapport détaillé (F1 par tag) pour la meilleure configuration sur CorBaMa
best_config = max(results, key=lambda k: results[k]["acc_corbama"])
print(f"Meilleure configuration sur CorBaMa : {best_config}\n")

if best_config.startswith("Broadcast"):
    model_best = model_bc_v2 if "v2" in best_config else model_bc_v1
    sp_best = sp_v2 if "v2" in best_config else sp_v1
    _, report = evaluate_subword_model_word_level(model_best, corbama_data, sp_best, id2label, "broadcast", device)
else:
    model_best = model_bio_v2 if "v2" in best_config else model_bio_v1
    sp_best = sp_v2 if "v2" in best_config else sp_v1
    _, report = evaluate_subword_model_word_level(model_best, corbama_data, sp_best, id2label_bio, "bio", device)

print(report)

Meilleure configuration sur CorBaMa : BIO v1

              precision    recall  f1-score   support

         ADJ       0.99      0.10      0.19     80453
         ADV       0.40      0.19      0.26     23045
         AUX       0.69      0.75      0.72    341048
        CONJ       0.61      0.54      0.57    101866
         DET       0.40      0.49      0.44     79879
         NOM       0.54      0.80      0.65    635374
        PART       0.45      0.24      0.31     55415
       POSTP       0.87      0.47      0.61    240354
        PRON       0.85      0.75      0.80    361139
       PUNCT       0.96      0.95      0.96    389376
       VERBE       0.50      0.39      0.44    323993

    accuracy                           0.67   2631942
   macro avg       0.66      0.52      0.54   2631942
weighted avg       0.70      0.67      0.66   2631942



# ***Interprétation***
Quatre configurations comparées sur un pied d'égalité (même corpus de test réel, même méthode d'évaluation au niveau mot) : Broadcast v1/v2 et BIO v1/v2. Regarder ensemble l'accuracy sur notre propre test (v2 ne doit pas être pire que v1 ici) et sur CorBaMa (le vrai test), plus la fragmentation